Q 1.You are appending 1M (10L) rows to delta table, after writing 6L records, the job failed due to cluster error. What will happen to the data already written, Will we see 6L rows in table?

A. We will not see any of those 6L rows in Delta table

Why ?

Delta Lake provides ATOMICITY as part of ACID transaction - meaning either all the data is committed or none of it is.

---------------------
So what happens behind the scenes for Q1

* When we run a write like (df.write.format('delta').mode('append').saveAsTable('delta_table_name')), 
  Spark stages the data in a temporary location.

* Only after all data is written successfully, Delta Lake commits the transaction by:
  * Writing a new JSON commit file in the _delta_log/ directory
  * Making the new data visible to readers

* If the job fails before the commit, the transaction is not logged and partial data files(like the 6L records are not part of table's current snapshot)


POINT 1. temporary location == delta table storage location
      2. orphaned files can be cleaned up by VACUUM

-------

Q2. We are ingesting 1000 rows into delta table, 999 rows are valid but 1 has a wrong data type ( e.g., a string in column expecting double). What happens during the write, how do you handle it?

A. By default, Spark (and Delta Lake) follows schema enforcement - meaning : If any row violates the schema, the entire write operation fails ( None of 999 valid rows will be written)

-------

How to handle this Q2 situation ?

Let Spark skip the bad records and write the valid ones:

df.write.format('delta').option('badRecordsPath','/tmp/badrecords/').mode('append').saveAsTable('table_name')

we can give dbfs or s3 path